# **LABORATORIO N 10 - Migracion de Datos y ETL**

**Curso:** Fundamentos de Gestion de Datos  
**Docente:** Pilar Rocio Sayan Mejia  
**Semana:** 10  
**Tema:** Pipeline ETL (Extract, Transform, Load) en Python

---

## **Actividad 1: Revision de Conceptos**

Complete la siguiente tabla con sus propias palabras:

| Concepto | Definicion |
|----------|------------|
| 1. Que es ETL? | |
| 2. Diferencia entre ETL y ELT | |
| 3. Que es la fase Extract? | |
| 4. Que es la fase Transform? | |
| 5. Que es la fase Load? | |
| 6. Fases de una migracion de datos | |
| 7. Principales desafios en migracion | |
| 8. Buenas practicas en ETL | |
| 9. Que es un pipeline de datos? | |
| 10. Que son las Stored Procedures? | |
| 11. Como validar integridad de datos? | |
| 12. Diferencia entre migracion e integracion | |

---

## **Actividad 2: Desarrollo Practico - Pipeline ETL**

En esta actividad implementaremos un pipeline ETL completo.

### **Paso 1: Configuracion del entorno**

In [ ]:
# Importar librerias necesarias
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("Librerias importadas correctamente")

### **Paso 2: Crear base de datos fuente de ejemplo**

In [ ]:
# Crear base de datos fuente con datos de ejemplo
conn_source = sqlite3.connect(':memory:')

# Crear tabla de ventas
conn_source.execute('''
    CREATE TABLE ventas (
        id INTEGER PRIMARY KEY,
        producto TEXT,
        cantidad INTEGER,
        precio REAL,
        fecha TEXT,
        cliente_id INTEGER,
        region TEXT
    )
''')

# Insertar datos de ejemplo (con algunos problemas de calidad)
np.random.seed(42)
n_records = 500

productos = ['Laptop', 'Mouse', 'Teclado', 'Monitor', 'Audifonos']
regiones = ['Norte', 'Sur', 'Este', 'Oeste', 'Centro']

data = []
for i in range(1, n_records + 1):
    # Introducir algunos datos nulos y duplicados para simular problemas
    if i % 50 == 0:
        data.append((i, None, np.random.randint(1, 20), np.random.uniform(10, 500), '2026-01-15', i % 100, None))
    else:
        data.append((
            i,
            np.random.choice(productos),
            np.random.randint(1, 20),
            round(np.random.uniform(10, 500), 2),
            f'2026-0{np.random.randint(1,4)}-{np.random.randint(10,28):02d}',
            i % 100,
            np.random.choice(regiones)
        ))

# Agregar algunos duplicados
data.extend(data[:10])

conn_source.executemany('INSERT INTO ventas VALUES (?, ?, ?, ?, ?, ?, ?)', data)
conn_source.commit()

print("Base de datos fuente creada")
df_check = pd.read_sql_query('SELECT * FROM ventas', conn_source)
print(f"Total registros en fuente: {len(df_check)}")

### **Paso 3: Fase EXTRACT - Extraccion de datos**

In [ ]:
# FASE EXTRACT
print("="*60)
print("FASE EXTRACT - Extraccion de datos")
print("="*60)

# Extraer datos de la tabla origen
query_extract = 'SELECT * FROM ventas'
df_extracted = pd.read_sql_query(query_extract, conn_source)

print(f"\nRegistros extraidos: {len(df_extracted)}")
print(f"Columnas: {list(df_extracted.columns)}")
print(f"\nTipos de datos:")
print(df_extracted.dtypes)
print(f"\nPrimeras 5 filas:")
df_extracted.head()

In [ ]:
# Diagnostico inicial de calidad
print("Diagnostico de calidad de datos extraidos:")
print(f"\nValores nulos por columna:")
print(df_extracted.isnull().sum())
print(f"\nDuplicados: {df_extracted.duplicated().sum()}")

**Pregunta 1:** Cuantos registros y columnas tiene tu dataset extraido? Describe brevemente su contenido.

**Respuesta:** _Escribe tu respuesta aqui_

### **Paso 4: Fase TRANSFORM - Transformacion de datos**

In [ ]:
# FASE TRANSFORM
print("="*60)
print("FASE TRANSFORM - Transformacion de datos")
print("="*60)

# Crear copia para transformar
df_transformed = df_extracted.copy()

In [ ]:
# Transformacion 1: Limpieza de datos
def transform_limpiar(df):
    """Elimina duplicados y maneja valores nulos"""
    print("\nTransformacion 1: Limpieza")
    print(f"  - Registros antes: {len(df)}")
    
    # Eliminar duplicados
    df = df.drop_duplicates()
    print(f"  - Duplicados eliminados")
    
    # Rellenar valores nulos
    df['producto'] = df['producto'].fillna('Sin especificar')
    df['region'] = df['region'].fillna('No asignada')
    df['precio'] = df['precio'].fillna(df['precio'].median())
    df['cantidad'] = df['cantidad'].fillna(1)
    print(f"  - Valores nulos imputados")
    
    print(f"  - Registros despues: {len(df)}")
    return df

df_transformed = transform_limpiar(df_transformed)

In [ ]:
# Transformacion 2: Agregacion de columnas
def transform_agregar(df):
    """Crea nuevas columnas derivadas"""
    print("\nTransformacion 2: Agregacion")
    
    # Calcular total de venta
    df['total_venta'] = df['cantidad'] * df['precio']
    print("  - Columna 'total_venta' creada")
    
    # Categorizar por monto
    df['categoria_venta'] = pd.cut(df['total_venta'], 
                                    bins=[0, 100, 500, 1000, float('inf')],
                                    labels=['Pequena', 'Mediana', 'Grande', 'Mayorista'])
    print("  - Columna 'categoria_venta' creada")
    
    return df

df_transformed = transform_agregar(df_transformed)

In [ ]:
# Transformacion 3: Enriquecimiento
def transform_enriquecer(df):
    """Agrega metadata y derivados temporales"""
    print("\nTransformacion 3: Enriquecimiento")
    
    # Agregar fecha de carga
    df['fecha_carga'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print("  - Columna 'fecha_carga' agregada")
    
    # Convertir fecha a datetime
    df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce')
    df['mes'] = df['fecha'].dt.month
    df['anio'] = df['fecha'].dt.year
    print("  - Columnas temporales creadas")
    
    return df

df_transformed = transform_enriquecer(df_transformed)

In [ ]:
# Mostrar resultado de transformaciones
print("\n" + "="*60)
print("RESULTADO DE TRANSFORMACIONES")
print("="*60)
print(f"\nRegistros finales: {len(df_transformed)}")
print(f"Columnas finales: {list(df_transformed.columns)}")
print(f"\nValores nulos despues de transformar:")
print(df_transformed.isnull().sum())
print(f"\nPrimeras 5 filas transformadas:")
df_transformed.head()

**Pregunta 2:** Que transformaciones aplicaste a los datos? Por que elegiste esas transformaciones?

**Respuesta:** _Escribe tu respuesta aqui_

### **Paso 5: Fase LOAD - Carga de datos**

In [ ]:
# FASE LOAD
print("="*60)
print("FASE LOAD - Carga de datos")
print("="*60)

# Crear nueva base de datos destino
conn_target = sqlite3.connect(':memory:')

# Cargar datos transformados
df_transformed.to_sql('ventas_transformadas', conn_target, if_exists='replace', index=False)

print(f"\nDatos cargados exitosamente")
print(f"Tabla: ventas_transformadas")
print(f"Registros: {len(df_transformed)}")

**Pregunta 3:** Que diferencias hay entre la tabla origen y la tabla destino? Muestra una comparacion.

**Respuesta:** _Escribe tu respuesta aqui_

### **Paso 6: Validacion del pipeline ETL**

In [ ]:
# VALIDACION DEL PIPELINE
print("="*60)
print("VALIDACION DEL PIPELINE ETL")
print("="*60)

# Validar cantidad de registros
count_source = len(df_extracted)
count_target = len(df_transformed)

print(f"\n1. Validacion de registros:")
print(f"   - Registros fuente: {count_source}")
print(f"   - Registros destino: {count_target}")
print(f"   - Diferencia: {count_source - count_target} (duplicados eliminados)")

# Validar columnas
cols_source = set(df_extracted.columns)
cols_target = set(df_transformed.columns)
new_cols = cols_target - cols_source

print(f"\n2. Validacion de columnas:")
print(f"   - Columnas originales: {len(cols_source)}")
print(f"   - Columnas finales: {len(cols_target)}")
print(f"   - Nuevas columnas: {new_cols}")

# Validar valores nulos
nulls_source = df_extracted.isnull().sum().sum()
nulls_target = df_transformed.isnull().sum().sum()

print(f"\n3. Validacion de calidad:")
print(f"   - Valores nulos fuente: {nulls_source}")
print(f"   - Valores nulos destino: {nulls_target}")

# Verificar datos cargados
df_verify = pd.read_sql_query('SELECT * FROM ventas_transformadas LIMIT 5', conn_target)
print(f"\n4. Verificacion de datos cargados:")
display(df_verify)

**Pregunta 4:** El pipeline ETL se ejecuto correctamente? Como verificaste la integridad de los datos?

**Respuesta:** _Escribe tu respuesta aqui_

---

## **Actividad 3: Caso de Estudio - Pipeline ETL del Proyecto Grupal**

Implementa un pipeline ETL sobre tu base de datos del proyecto grupal.

In [ ]:
# ESPACIO PARA TU PIPELINE ETL DEL PROYECTO

# # Conectar a tu base de datos fuente
# conn_proyecto = sqlite3.connect('tu_proyecto.db')

# # EXTRACT
# df_proyecto = pd.read_sql_query('SELECT * FROM tu_tabla', conn_proyecto)
# print(f"Registros extraidos: {len(df_proyecto)}")

# # TRANSFORM
# # Aplica tus transformaciones...

# # LOAD
# # Carga los datos transformados...

**Pregunta A:** Describe las fuentes de datos de tu proyecto. De donde provienen los datos originales?

**Respuesta:** _Escribe tu respuesta aqui_

**Pregunta B:** Que transformaciones aplicaste? Justifica cada una de ellas.

**Respuesta:** _Escribe tu respuesta aqui_

**Pregunta C:** Muestra el antes y despues del pipeline con df.shape. Hubo perdida de datos?

**Respuesta:** _Escribe tu respuesta aqui_

---

## **Conclusiones**

Escribe tus conclusiones tecnicas y de aprendizaje:

1. _Escribe tu primera conclusion aqui_

2. _Escribe tu segunda conclusion aqui_

3. _Escribe tu tercera conclusion aqui_

In [ ]:
# Cerrar conexiones
conn_source.close()
conn_target.close()
print("Conexiones cerradas correctamente")